# Reddit RSS 抓取数据长什么样？（结构 + 具体内容）

这个 Notebook 用你的 `RedditRSSClient` 真跑一次 `fetch_posts()`，然后展示：
- 返回值整体结构（list[dict]）
- 单条 post 的 keys、字段类型
- `text` 原始内容（通常包含 HTML）
- 转成 DataFrame 后的列、示例行
- 可选：清理 HTML 后的文本对比

✅ 需要：本 ipynb 与 `reddit_rss_client.py` 在同一目录（或自行修改 import 路径）。


In [1]:
import sys
import os
sys.path.append("/Users/chenjiexin/Desktop/学习/financial-sentiment-analysis")
from reddit_rss_client import RedditRSSClient



In [2]:
import json
import re
from pathlib import Path
from pprint import pprint

import pandas as pd

# 如果你的文件名就是 reddit_rss_client.py，且与本 ipynb 同目录：
from reddit_rss_client import RedditRSSClient

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 50)


In [3]:
# 1) 初始化 client
client = RedditRSSClient(config_path='config.json')

# 2) 抓取数据（你可以改 max_results / query / start_date / end_date）
posts = client.fetch_posts(
    query=None,          # None 就用 default_query
    max_results=8,       # 建议先小一点方便观察
    start_date=None,
    end_date=None
)

print('type(posts):', type(posts))
print('len(posts):', len(posts))
print('type(posts[0]):', type(posts[0]) if posts else None)


type(posts): <class 'list'>
len(posts): 8
type(posts[0]): <class 'dict'>


In [4]:
# 3) 看“单条 post”到底长什么样（完整 dict）
if posts:
    print('--- One raw post dict ---')
    pprint(posts[0])
else:
    print('No posts fetched. Try increasing max_results or check network.')


--- One raw post dict ---
{'author': '/u/jkim2077',
 'author_id': '/u/jkim2077',
 'created_at': '2026-01-25T16:32:16+00:00',
 'id': 'reddit_t3_1qmnte9',
 'link': 'https://www.reddit.com/r/stocks/comments/1qmnte9/memory_demands_reality/',
 'metrics': {},
 'reddit_id': 't3_1qmnte9',
 'subreddit': 'stocks',
 'text': 'Memory demands reality',
 'timezone': 'UTC',
 'title': 'Memory demands reality',
 'url': 'https://www.reddit.com/r/stocks/comments/1qmnte9/memory_demands_reality/'}


In [5]:
# 4) 查看字段结构：keys + 每个字段的数据类型
def schema_of_one(post: dict):
    return {k: type(v).__name__ for k, v in post.items()}

if posts:
    print('--- Keys ---')
    print(list(posts[0].keys()))
    print('\n--- Field types ---')
    pprint(schema_of_one(posts[0]))


--- Keys ---
['id', 'reddit_id', 'url', 'subreddit', 'title', 'text', 'author', 'author_id', 'created_at', 'timezone', 'link', 'metrics']

--- Field types ---
{'author': 'str',
 'author_id': 'str',
 'created_at': 'str',
 'id': 'str',
 'link': 'str',
 'metrics': 'dict',
 'reddit_id': 'str',
 'subreddit': 'str',
 'text': 'str',
 'timezone': 'str',
 'title': 'str',
 'url': 'str'}


In [6]:
# 5) 转成 DataFrame：看表格形式（前端/数据库通常更喜欢这种）
df = pd.DataFrame(posts)
print('df.shape:', df.shape)
df.head(5)


df.shape: (8, 12)


,id,reddit_id,url,subreddit,title,text,author,author_id,created_at,timezone,link,metrics
0,reddit_t3_1qmnte9,t3_1qmnte9,https://www.reddit.com/r/stocks/comments/1qmnte9/memory_demands_reality/,stocks,Memory demands reality,Memory demands reality,/u/jkim2077,/u/jkim2077,2026-01-25T16:32:16+00:00,UTC,https://www.reddit.com/r/stocks/comments/1qmnte9/memory_demands_reality/,{}
1,reddit_t3_1qmliej,t3_1qmliej,https://www.reddit.com/r/stocks/comments/1qmliej/will_usar_rise_with_the_us_government_buying_10/,stocks,Will USAR rise with the US government buying 10% of the company tomorrow?,Will USAR rise with the US government buying 10% of the company tomorrow?,/u/Deeizzle23,/u/Deeizzle23,2026-01-25T15:07:34+00:00,UTC,https://www.reddit.com/r/stocks/comments/1qmliej/will_usar_rise_with_the_us_government_buying_10/,{}
2,reddit_t3_1qmuu6e,t3_1qmuu6e,https://www.reddit.com/r/StockMarket/comments/1qmuu6e/just_found_this_portfolio/,StockMarket,Just found this portfolio 😬…,Just found this portfolio 😬…,/u/kabirsbhutani,/u/kabirsbhutani,2026-01-25T20:44:39+00:00,UTC,https://www.reddit.com/r/StockMarket/comments/1qmuu6e/just_found_this_portfolio/,{}
3,reddit_t3_1qms5o5,t3_1qms5o5,https://www.reddit.com/r/StockMarket/comments/1qms5o5/the_shortterm_safety_of_heavily_beaten_down_blue/,StockMarket,The short-term safety of heavily beaten down blue chips and their potential to rally are underrated.,The short-term safety of heavily beaten down blue chips and their potential to rally are underrated.,/u/lies_are_comforting,/u/lies_are_comforting,2026-01-25T19:07:47+00:00,UTC,https://www.reddit.com/r/StockMarket/comments/1qms5o5/the_shortterm_safety_of_heavily_beaten_down_blue/,{}
4,reddit_t3_1qmu24p,t3_1qmu24p,https://www.reddit.com/r/investing/comments/1qmu24p/i_need_some_401k_reallocation_advice/,investing,I need some 401k Reallocation advice,I need some 401k Reallocation advice,/u/BigTruckSmallPP,/u/BigTruckSmallPP,2026-01-25T20:15:54+00:00,UTC,https://www.reddit.com/r/investing/comments/1qmu24p/i_need_some_401k_reallocation_advice/,{}


In [7]:
# 6) 最关键：text 字段的“具体内容”长什么样？
# 注意：RSS summary 常常带 HTML（<p>、<a>...），你现在代码只是 unescape 了，并没有 strip 标签

if not df.empty:
    for i in range(min(3, len(df))):
        print('\n' + '='*90)
        print(f'[{i}] subreddit={df.loc[i, "subreddit"]} | created_at={df.loc[i, "created_at"]}')
        print('title:', df.loc[i, 'title'][:200])
        print('\ntext (first 800 chars):\n')
        print(df.loc[i, 'text'][:800])



[0] subreddit=stocks | created_at=2026-01-25T16:32:16+00:00
title: Memory demands reality

text (first 800 chars):

Memory demands reality

[1] subreddit=stocks | created_at=2026-01-25T15:07:34+00:00
title: Will USAR rise with the US government buying 10% of the company tomorrow?

text (first 800 chars):

Will USAR rise with the US government buying 10% of the company tomorrow?

[2] subreddit=StockMarket | created_at=2026-01-25T20:44:39+00:00
title: Just found this portfolio 😬…

text (first 800 chars):

Just found this portfolio 😬…


In [8]:
# 7) （可选）清理 HTML：对比“原始 text” vs “去标签后的纯文本”

TAG_RE = re.compile(r'<[^>]+>')

def strip_html(s: str) -> str:
    if s is None:
        return ''
    # 去标签 + 压缩空白
    s2 = TAG_RE.sub(' ', s)
    s2 = re.sub(r'\s+', ' ', s2).strip()
    return s2

if not df.empty:
    df['text_clean'] = df['text'].apply(strip_html)
    df[['subreddit','title','text','text_clean']].head(3)


In [9]:
# 8) 保存这次抓到的“原始结果”到本地 JSON（用于 debug/对照数据库写入）

out_path = Path('sample_posts_raw.json')
out_path.write_text(json.dumps(posts, ensure_ascii=False, indent=2))
print('Saved:', out_path.resolve())

# 如果你也想保存表格版
if not df.empty:
    df.to_csv('sample_posts_table.csv', index=False)
    print('Saved:', Path('sample_posts_table.csv').resolve())


Saved: /Users/chenjiexin/Desktop/学习/financial-sentiment-analysis/backend/sample_posts_raw.json
Saved: /Users/chenjiexin/Desktop/学习/financial-sentiment-analysis/backend/sample_posts_table.csv
